# Failure notification (mail)

Reads the failures out of `monitoring.unified_run_history` - pipelines, semantic model refreshes and MLV refreshes alike - mails each one **once**, and keeps the proof it went out. The merge notebook (`load_unified_run_history.ipynb`) stays free of mail; everything mail-related lives here.

Owns two tables (created by cell 3 if missing) plus a view:

- `monitoring.run_alert_state` - one row per run that has been picked up for notification: `sent_mail`, `sent_mail_at_utc`, `mail_status`, `mail_attempt_count`, `mail_batch_id`, `mail_recipients`, `mail_error`
- `monitoring.mail_send_log` - one row per attempted email: recipients, subject, the exact HTML body, queued / sent timestamps, error, and the Fabric activity that built it
- `monitoring.vw_run_alert_status` - every failure with its notification state, for "what broke and was anyone told"

A notebook has no mailbox, so this one **picks and records** and the pipeline's Office365Outlook activity sends.

## One mail per window, every failure in it
A run of this notebook sends **at most one email**, containing every failure it picked - pipelines, semantic models and MLVs together, as one table sorted by source and time. `MAIL_COOLDOWN_MINUTES` (30 by default) then holds the next mail back: on a 15-minute schedule the run that lands inside the window prints what is waiting and claims nothing, so those failures ride along in the next digest instead of arriving as separate mails. Set it to `0` to mail as soon as anything is found, or to `15` to match the schedule exactly.

The cooldown is measured from the last batch handed to the mail activity, whatever became of it, so a broken mailbox cannot turn into a send-per-run loop.

## Pipeline wiring
```
notify_run_failures (MAIL_ENABLED = True, ALERT_RECIPIENTS = @pipeline().parameters.alert_recipients)
   |
   +-- If  @greater(json(activity('notify_run_failures').output.result.exitValue).alert_count, 0)
         |
         +-- Office365Outlook "Send an email"
               To      @pipeline().parameters.alert_recipients
               Subject @json(activity('notify_run_failures').output.result.exitValue).subject
               Body    @json(activity('notify_run_failures').output.result.exitValue).message_html
               |
               +-- on Success -> notify_run_failures again:
               |      MAIL_MODE = confirm
               |      CONFIRM_BATCH_ID = @json(activity('notify_run_failures').output.result.exitValue).mail_batch_id
               |      CONFIRM_STATUS   = Sent
               |
               +-- on Failure -> the same, with CONFIRM_STATUS = Failed
                      CONFIRM_ERROR = @{activity('Send an email').error.message}
```

The confirm step is optional, and `ASSUME_SENT_ON_EXIT` says which way it is wired:

- **`True` (default)** - no confirm activity. Runs are stamped `sent_mail = TRUE, mail_status = 'SentUnconfirmed'` the moment the payload leaves, so a run can never be mailed twice. The name is the caveat: the send was handed over, not witnessed. If you wire only the *failure* path of the mail activity to the confirm step, a failed send flips those rows back to `Failed` and they are retried - the best of both.
- **`False`** - both confirm paths wired. `sent_mail` is then set only by a confirmed send, and a batch that never gets confirmed (the pipeline died between the two activities) is reaped back to `Failed` after `MAIL_QUEUE_TIMEOUT_MINUTES` and retried, up to `MAIL_MAX_ATTEMPTS`, then parked as `GaveUp`. **Do not set this without the confirm activity**: every batch would time out and be re-mailed.

`mail_status` values: `Queued` -> `Sent` | `SentUnconfirmed` | `Failed` -> `GaveUp`.

## How a failure is identified as "not mailed yet"
`unified_run_history` carries no mail columns - the identity is `run_key`, stable for the life of a run (`Pipeline:<run_id>`, `SemanticModel:<model_id>:<request_id>`, `MLV:<iteration_id>`), and `run_alert_state` holds one row per run_key ever picked up. The pick is an anti-join against it:

```
s.run_key IS NULL                                   -- never picked up -> mail it
NOT s.sent_mail AND s.mail_status = 'Failed'        -- send failed      -> retry it
    AND s.mail_attempt_count < MAIL_MAX_ATTEMPTS
anything else (Queued, Sent, SentUnconfirmed, GaveUp)                  -> skipped for good
```

The state row is written **before** the payload leaves the notebook, so the next run - 15 minutes later - already sees the run claimed and skips it, confirmed or not. A run is mailed once; the only path to a second mail is a send explicitly reported as failed, which is a retry rather than a duplicate. A status change on an already-mailed run (the merge finding more failed activities, say) does not re-open it.

Re-merging, backfilling with `FULL_RELOAD`, even dropping and rebuilding `unified_run_history` changes none of this: `run_key` does not depend on load time, and this state table is the memory - never truncate it.

## Schedule
Built for **every 15 minutes**, as the last step of the pipeline that runs the three loaders and then `load_unified_run_history` - a failure can only be mailed once it has been merged. Set that pipeline's concurrency to **1**: two overlapping runs could both see the same unclaimed failure.

Alerts are only as fresh as the slowest loader - if the pipeline-run loader still runs hourly, pipeline failures are mailed up to an hour late however often this notebook runs.

`ALERT_MAX_AGE_HOURS` is the backstop: a failure that ended longer ago is never mailed, so a backfill or a long outage cannot flood the inbox with history.

## Running it by hand
With `MAIL_ENABLED = False` (the default) nothing is claimed, logged or sent: the notebook picks the failures that *would* be mailed and prints the subject and body, so the message can be checked before the pipeline is armed.

In [ ]:
# ---- Config (parameter cell: the monitor pipeline overrides MAIL_ENABLED, ALERT_RECIPIENTS and the CONFIRM_* values) ----

# Source: the merged run history
UNIFIED_TABLE = "monitoring.unified_run_history"

# Owned by this notebook
STATE_TABLE    = "monitoring.run_alert_state"
MAIL_LOG_TABLE = "monitoring.mail_send_log"
STATUS_VIEW    = "monitoring.vw_run_alert_status"

LOCAL_TZ       = "Asia/Singapore"
LOCAL_TZ_LABEL = "SGT"

# ---- What to alert on ----
ALERT_STATUSES = ["Failed", "CompletedWithErrors"]     # effective_status values worth an email
ALERT_SOURCES  = ["Pipeline", "SemanticModel", "MLV"]  # drop a source to stop mailing it
EXCLUDE_ITEM_CONTAINS = ["obsolete"]                   # skip items whose name contains these (case-insensitive)
ALERT_MAX_AGE_HOURS   = 24    # never mail a failure that ended longer ago than this (so a backfill stays quiet)

# A run is mailed once: it is claimed in run_alert_state before the payload leaves, so the
# next run (15 min later) skips it. Only an explicitly failed send is ever picked again.

# ---- Mail ----
MAIL_MODE    = "queue"        # queue = pick, claim, hand the payload to the pipeline | confirm = record the result
MAIL_ENABLED = False          # the pipeline passes True; by hand it stays a preview
ALERT_RECIPIENTS = ""         # logged with the batch; the pipeline's Outlook activity does the sending

# One mail per window: every failure found goes in the same body, and no second mail goes
# out until this many minutes after the last one. Failures found inside the window are left
# unclaimed and ride along in the next mail. 0 = mail on every run that finds something.
MAIL_COOLDOWN_MINUTES = 30

MAIL_MAX_ATTEMPTS          = 3     # stop retrying a run after this many failed sends
MAIL_QUEUE_TIMEOUT_MINUTES = 60    # a Queued batch with no confirmation after this is retried
MAIL_MAX_RUNS_IN_BODY      = 20    # rows in the table, then "...and N more"

# True  = no confirm activity in the pipeline: stamp sent_mail + 'SentUnconfirmed' at queue
#         time, so nothing is ever mailed twice (a confirmed failure still reopens it).
# False = both confirm paths are wired: sent_mail is set only by a confirmed send, and an
#         unconfirmed batch is retried after MAIL_QUEUE_TIMEOUT_MINUTES. Setting this
#         without the confirm activity re-mails every failure once an hour.
ASSUME_SENT_ON_EXIT = True

# ---- Confirm mode (set by the pipeline on the success / failure path of the mail activity) ----
CONFIRM_BATCH_ID = ""
CONFIRM_STATUS   = "Sent"     # Sent | Failed
CONFIRM_ERROR    = ""

In [ ]:
# ---- Targets: create if missing ----
spark.sql("CREATE SCHEMA IF NOT EXISTS monitoring")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {STATE_TABLE} (
    run_key             STRING    NOT NULL COMMENT '-> unified_run_history.run_key; one row per notified run',
    source_type         STRING             COMMENT 'Pipeline | SemanticModel | MLV',
    item_name           STRING             COMMENT 'Pipeline / semantic model / lakehouse the run belongs to',
    effective_status    STRING             COMMENT 'Status that triggered the notification',
    run_end_time_utc    TIMESTAMP          COMMENT 'When the run itself ended',
    sent_mail           BOOLEAN   NOT NULL COMMENT 'TRUE once the mail activity confirmed the send (or at queue time if ASSUME_SENT_ON_EXIT)',
    sent_mail_at_utc    TIMESTAMP          COMMENT 'When that happened',
    mail_status         STRING    NOT NULL COMMENT 'Queued | Sent | SentUnconfirmed | Failed | GaveUp',
    mail_attempt_count  INT       NOT NULL COMMENT 'Batches this run has been in; capped by MAIL_MAX_ATTEMPTS',
    mail_batch_id       STRING             COMMENT '-> mail_send_log.mail_batch_id',
    mail_recipients     STRING             COMMENT 'To: the batch was queued for',
    mail_error          STRING             COMMENT 'Why the last send failed or timed out',
    first_queued_at_utc TIMESTAMP NOT NULL COMMENT 'First time this run was put in a batch',
    last_queued_at_utc  TIMESTAMP NOT NULL COMMENT 'Most recent batch',
    updated_at_utc      TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Notification state per run - the sent_mail flag and its timestamp'
TBLPROPERTIES (
    'delta.parquet.vorder.enabled'   = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {MAIL_LOG_TABLE} (
    mail_log_id     STRING    NOT NULL COMMENT 'uuid4 of this log row',
    mail_batch_id   STRING    NOT NULL COMMENT 'One batch = one email; -> run_alert_state.mail_batch_id',
    mail_status     STRING    NOT NULL COMMENT 'Queued -> Sent | SentUnconfirmed | Failed | TimedOut',
    run_count       INT       NOT NULL COMMENT 'Runs covered by the email',
    source_types    STRING             COMMENT 'Sources in the batch, e.g. "MLV, Pipeline"',
    run_keys        STRING             COMMENT 'The run_key values in the batch, comma separated',
    recipients      STRING,
    subject         STRING,
    body_html       STRING             COMMENT 'Exactly what was handed to the mail activity',
    queued_at_utc   TIMESTAMP NOT NULL,
    sent_at_utc     TIMESTAMP          COMMENT 'Set when the send is confirmed',
    error_message   STRING,
    notebook_run_id STRING             COMMENT 'Fabric activity / job id that built the batch',
    updated_at_utc  TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Mail send log - one row per attempted email'
TBLPROPERTIES (
    'delta.parquet.vorder.enabled'   = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
)
""")

# Every failure with its notification state - including the ones nobody was told about.
spark.sql(f"""
CREATE OR REPLACE VIEW {STATUS_VIEW} AS
SELECT u.source_type,
       u.item_name,
       u.run_name,
       u.effective_status,
       from_utc_timestamp(u.start_time_utc, '{LOCAL_TZ}')                      AS start_time_sgt,
       from_utc_timestamp(u.end_time_utc,   '{LOCAL_TZ}')                      AS end_time_sgt,
       CAST(from_utc_timestamp(u.start_time_utc, '{LOCAL_TZ}') AS DATE)        AS run_date_sgt,
       ROUND(u.duration_ms / 60000.0, 2)                                       AS duration_minutes,
       u.child_failed,
       u.failed_children,
       u.error_code,
       COALESCE(u.error_detail, u.error_message)                               AS error_detail,
       COALESCE(s.sent_mail, FALSE)                                            AS sent_mail,
       from_utc_timestamp(s.sent_mail_at_utc, '{LOCAL_TZ}')                    AS sent_mail_at_sgt,
       COALESCE(s.mail_status, 'NotPicked')                                    AS mail_status,
       COALESCE(s.mail_attempt_count, 0)                                       AS mail_attempt_count,
       s.mail_batch_id,
       s.mail_recipients,
       s.mail_error,
       u.run_key
FROM {UNIFIED_TABLE} u
LEFT JOIN {STATE_TABLE} s ON s.run_key = u.run_key
WHERE u.is_failure
""")

print(f"{STATE_TABLE}, {MAIL_LOG_TABLE} and {STATUS_VIEW} ready")

In [ ]:
# ---- Helpers ----
import html
import json
import uuid
from datetime import datetime, timedelta, timezone

from notebookutils import mssparkutils

spark.conf.set("spark.sql.session.timeZone", "UTC")
now = datetime.now(timezone.utc)


def sq(value):
    """Quote a value for inline SQL; None and '' become NULL."""
    if value is None or value == "":
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"


def ts(dt):
    return f"TIMESTAMP '{dt:%Y-%m-%d %H:%M:%S}'"


def sql_list(values):
    return ", ".join(sq(v) for v in values) or "NULL"


def notebook_run_id():
    """The Fabric activity / job id of this run, so a log row points back at it."""
    try:
        ctx = mssparkutils.runtime.context
        return ctx.get("activityId") or ctx.get("currentNotebookId")
    except Exception:
        return None


try:
    from zoneinfo import ZoneInfo
    LOCAL_ZONE = ZoneInfo(LOCAL_TZ)
except Exception:                        # no tz database in the image: print UTC instead
    LOCAL_ZONE = timezone.utc


def as_utc(dt):
    """Spark hands back naive datetimes in the session timezone, which is UTC here."""
    if dt is None:
        return None
    return dt if dt.tzinfo else dt.replace(tzinfo=timezone.utc)


def fmt_local(dt, fmt="%Y-%m-%d %H:%M"):
    return f"{as_utc(dt).astimezone(LOCAL_ZONE):{fmt}} {LOCAL_TZ_LABEL}" if dt else "unknown"


# ---- Email: one digest for every failure in the window ----
SOURCE_ORDER  = ["Pipeline", "SemanticModel", "MLV"]
SOURCE_LABELS = {"Pipeline":      ("data pipeline", "data pipelines"),
                 "SemanticModel": ("semantic model refresh", "semantic model refreshes"),
                 "MLV":           ("materialized lake view", "materialized lake views")}
SOURCE_SHORT  = {"Pipeline": "Pipeline", "SemanticModel": "Semantic model", "MLV": "MLV"}
# Outlook ignores <style> blocks, so every rule is inline.
FONT   = "font-family:Segoe UI,Arial,Helvetica,sans-serif"
HEAD   = f"padding:6px 10px;border-bottom:1px solid #c8c6c4;text-align:left;font-weight:600;white-space:nowrap"
CELL   = "padding:6px 10px;border-bottom:1px solid #edebe9;vertical-align:top"
MUTED  = "color:#605e5c"
STATUS_COLORS = {"Failed": "#a4262c", "CompletedWithErrors": "#8a5700", "Cancelled": "#605e5c"}


def esc(value, limit=300):
    text = "" if value is None else str(value)
    return html.escape(text if len(text) <= limit else text[:limit] + "...")


def alert_html(rows, max_runs, last_mail_at=None):
    """One table for all the picked failures, whatever source they came from."""
    if not rows:
        return ""
    shown = rows[:max_runs]

    per_source = {s: sum(1 for r in rows if r.source_type == s) for s in SOURCE_ORDER}
    counts = ", ".join(f"{n} {SOURCE_LABELS[s][0] if n == 1 else SOURCE_LABELS[s][1]}"
                       for s, n in per_source.items() if n)
    ends = [r.end_time_local or r.start_time_local for r in rows
            if (r.end_time_local or r.start_time_local)]
    meta = [f"ran between {min(ends):%d %b %H:%M} and {max(ends):%d %b %H:%M} {LOCAL_TZ_LABEL}"
            if ends else "run times unknown"]
    if last_mail_at:
        meta.append(f"previous alert {fmt_local(last_mail_at, '%H:%M')}")

    parts = [f'<p style="{FONT};font-size:14px"><b>Fabric monitoring alert</b> &ndash; '
             f'{len(rows)} run(s) need attention ({counts})</p>',
             f'<p style="{FONT};font-size:12px;{MUTED}">{" &middot; ".join(meta)}</p>',
             f'<table cellpadding="0" cellspacing="0" border="0" '
             f'style="border-collapse:collapse;{FONT};font-size:13px;color:#201f1e">',
             '<tr style="background:#f3f2f1">' + "".join(
                 f'<th style="{HEAD}">{h}</th>' for h in
                 ("Source", "Item", "Status", f"Ran ({LOCAL_TZ_LABEL})", "Minutes",
                  "Failed part", "Error")) + "</tr>"]

    for r in shown:
        status = esc(r.effective_status)
        if r.mail_attempt_count:
            status += f' <span style="{MUTED};font-weight:400">(retry ' \
                      f'{r.mail_attempt_count + 1}/{MAIL_MAX_ATTEMPTS})</span>'
        item = f"<b>{esc(r.item_name or r.run_key, 60)}</b>"
        detail = r.run_name or r.invoke_type
        if detail:
            item += f'<br><span style="{MUTED};font-size:12px">{esc(detail, 50)}</span>'
        ran = r.end_time_local or r.start_time_local
        failed_part = esc(r.failed_children, 90) or (f"{r.child_failed} failed"
                                                     if r.child_failed else "&ndash;")
        cells = [
            esc(SOURCE_SHORT.get(r.source_type, r.source_type)),
            item,
            f'<span style="color:{STATUS_COLORS.get(r.effective_status, "#a4262c")};'
            f'font-weight:600">{status}</span>',
            f"{ran:%d %b %H:%M}" if ran else "&ndash;",
            f"{r.duration_minutes:.1f}" if r.duration_minutes is not None else "&ndash;",
            failed_part,
            f'<div style="max-width:460px">{esc(r.error_detail, 220)}</div>'
            if r.error_detail else "&ndash;",
        ]
        parts.append("<tr>" + "".join(f'<td style="{CELL}">{c}</td>' for c in cells) + "</tr>")

    if len(rows) > max_runs:
        parts.append(f'<tr><td colspan="7" style="{CELL};{MUTED}">'
                     f'...and {len(rows) - max_runs} more</td></tr>')
    parts.append("</table>")
    parts.append(f'<p style="{FONT};font-size:12px;{MUTED}">Mail state: <code>{STATUS_VIEW}</code>'
                 f' &middot; every run: <code>monitoring.vw_unified_run_health</code></p>')
    return "".join(parts)


def alert_subject(rows, max_len=200):
    """Subject naming what broke, so the inbox is readable without opening the mail."""
    if not rows:
        return ""
    names = ", ".join(sorted({(r.item_name or r.source_type) for r in rows}))
    subject = f"[Fabric] {len(rows)} run(s) need attention: {names}"
    return subject if len(subject) <= max_len else subject[:max_len - 3] + "..."

In [ ]:
# ---- Confirm mode: record what the mail activity did, and stop ----
# Rows already stamped SentUnconfirmed (ASSUME_SENT_ON_EXIT) are still open to a verdict
# here: confirmed upgrades them to Sent, a reported failure takes the flag back off.
if MAIL_MODE == "confirm":
    if not CONFIRM_BATCH_ID:
        raise ValueError("MAIL_MODE = 'confirm' needs CONFIRM_BATCH_ID")

    sent = CONFIRM_STATUS.strip().lower() == "sent"
    reason = CONFIRM_ERROR or "Mail activity reported a failure"
    awaiting = "(NOT sent_mail OR mail_status = 'SentUnconfirmed')"

    if sent:
        spark.sql(f"""
            UPDATE {STATE_TABLE}
               SET sent_mail        = TRUE,
                   sent_mail_at_utc = {ts(now)},
                   mail_status      = 'Sent',
                   mail_error       = NULL,
                   updated_at_utc   = {ts(now)}
             WHERE mail_batch_id = {sq(CONFIRM_BATCH_ID)} AND {awaiting}
        """)
    else:
        # The send did not happen: clear the flag so the next queue run retries this run,
        # until MAIL_MAX_ATTEMPTS. This is the only way a run is mailed more than once.
        spark.sql(f"""
            UPDATE {STATE_TABLE}
               SET sent_mail        = FALSE,
                   sent_mail_at_utc = NULL,
                   mail_status      = CASE WHEN mail_attempt_count >= {MAIL_MAX_ATTEMPTS}
                                           THEN 'GaveUp' ELSE 'Failed' END,
                   mail_error       = {sq(reason)},
                   updated_at_utc   = {ts(now)}
             WHERE mail_batch_id = {sq(CONFIRM_BATCH_ID)} AND {awaiting}
        """)

    spark.sql(f"""
        UPDATE {MAIL_LOG_TABLE}
           SET mail_status    = {sq('Sent' if sent else 'Failed')},
               sent_at_utc    = {ts(now) if sent else 'sent_at_utc'},
               error_message  = {sq(None if sent else reason)},
               updated_at_utc = {ts(now)}
         WHERE mail_batch_id = {sq(CONFIRM_BATCH_ID)}
    """)

    display(spark.sql(f"""
        SELECT run_key, source_type, item_name, effective_status,
               sent_mail, sent_mail_at_utc, mail_status, mail_attempt_count, mail_error
        FROM {STATE_TABLE}
        WHERE mail_batch_id = {sq(CONFIRM_BATCH_ID)}
        ORDER BY source_type, item_name
    """))
    print(f"Batch {CONFIRM_BATCH_ID} recorded as {'Sent' if sent else 'Failed'}")
    mssparkutils.notebook.exit(json.dumps({
        "alert_count":   0,
        "subject":       "",
        "message_html":  "",
        "mail_batch_id": CONFIRM_BATCH_ID,
        "mail_status":   "Sent" if sent else "Failed",
    }))

In [ ]:
# ---- Pick the failures that still need a mail ----
last_mail_at, cooldown_until, in_cooldown = None, None, False

if MAIL_MODE == "confirm":
    due = []                      # only reached on an interactive "run all" after the confirm cell
    print("confirm mode: nothing is picked")
else:
    if MAIL_ENABLED:
        queue_cutoff = now - timedelta(minutes=MAIL_QUEUE_TIMEOUT_MINUTES)

        if not ASSUME_SENT_ON_EXIT:
            # The one wiring mistake that causes duplicate mails: batches keep timing out
            # because no confirm step ever reports in, so every failure is sent again.
            log_state = spark.sql(f"""
                SELECT COUNT_IF(mail_status = 'TimedOut')          AS timed_out,
                       COUNT_IF(mail_status IN ('Sent', 'Failed')) AS confirmed
                FROM {MAIL_LOG_TABLE}
            """).collect()[0]
            if log_state.timed_out and not log_state.confirmed:
                print(f"WARNING: {log_state.timed_out} batch(es) timed out and no confirm step has ever"
                      " reported in.\n         The confirm activity looks unwired, so failures are being"
                      " re-mailed.\n         Wire it, or set ASSUME_SENT_ON_EXIT = True.")

        # Batches nobody ever confirmed - the pipeline died between the mail activity and
        # the confirm step - go back in the queue instead of sitting Queued forever.
        spark.sql(f"""
            UPDATE {STATE_TABLE}
               SET mail_status    = 'Failed',
                   mail_error     = 'No send confirmation within {MAIL_QUEUE_TIMEOUT_MINUTES} min',
                   updated_at_utc = {ts(now)}
             WHERE mail_status = 'Queued' AND NOT sent_mail
               AND last_queued_at_utc < {ts(queue_cutoff)}
        """)
        spark.sql(f"""
            UPDATE {MAIL_LOG_TABLE}
               SET mail_status    = 'TimedOut',
                   error_message  = COALESCE(error_message, 'No send confirmation'),
                   updated_at_utc = {ts(now)}
             WHERE mail_status = 'Queued' AND queued_at_utc < {ts(queue_cutoff)}
        """)

        # Runs that have used up their attempts stop being picked, so one bad recipient
        # cannot hold up every later alert.
        spark.sql(f"""
            UPDATE {STATE_TABLE}
               SET mail_status = 'GaveUp', updated_at_utc = {ts(now)}
             WHERE mail_status = 'Failed' AND NOT sent_mail
               AND mail_attempt_count >= {MAIL_MAX_ATTEMPTS}
        """)

    excluded = "".join(
        f"\n          AND COALESCE(LOWER(u.item_name), '') NOT LIKE {sq('%' + x.lower() + '%')}"
        for x in EXCLUDE_ITEM_CONTAINS)

    due = spark.sql(f"""
        SELECT u.run_key, u.source_type, u.item_name, u.run_name, u.invoke_type,
               u.effective_status,
               u.end_time_utc                                      AS run_end_time_utc,
               from_utc_timestamp(u.start_time_utc, '{LOCAL_TZ}')   AS start_time_local,
               from_utc_timestamp(u.end_time_utc,   '{LOCAL_TZ}')   AS end_time_local,
               ROUND(u.duration_ms / 60000.0, 1)                    AS duration_minutes,
               u.child_failed,
               u.failed_children,
               u.error_code,
               COALESCE(u.error_detail, u.error_message)            AS error_detail,
               COALESCE(s.mail_attempt_count, 0)                    AS mail_attempt_count
        FROM {UNIFIED_TABLE} u
        LEFT JOIN {STATE_TABLE} s ON s.run_key = u.run_key
        WHERE u.effective_status IN ({sql_list(ALERT_STATUSES)})
          AND u.source_type      IN ({sql_list(ALERT_SOURCES)})
          AND u.is_terminal
          AND COALESCE(u.end_time_utc, u.start_time_utc) >= {ts(now - timedelta(hours=ALERT_MAX_AGE_HOURS))}{excluded}
          AND (    s.run_key IS NULL                                    -- never picked up
               OR (NOT s.sent_mail AND s.mail_status = 'Failed'         -- a send that failed
                   AND s.mail_attempt_count < {MAIL_MAX_ATTEMPTS}))
        ORDER BY u.source_type, u.start_time_utc
    """).collect()

    retries = sum(1 for r in due if r.mail_attempt_count)
    print(f"{len(due)} failure(s) picked" + (f", {retries} of them a retry" if retries else ""))

    # One mail per window. The clock runs from the last batch handed to the mail activity,
    # whatever became of it, so a broken mailbox cannot turn into a send-per-run loop.
    last_mail_at = as_utc(spark.sql(f"SELECT MAX(queued_at_utc) AS t FROM {MAIL_LOG_TABLE}").collect()[0].t)
    if last_mail_at and MAIL_COOLDOWN_MINUTES:
        cooldown_until = last_mail_at + timedelta(minutes=MAIL_COOLDOWN_MINUTES)
        in_cooldown = now < cooldown_until
    print(f"Last mail {fmt_local(last_mail_at)}; cooldown {MAIL_COOLDOWN_MINUTES} min"
          + (f", next mail from {fmt_local(cooldown_until)}" if in_cooldown else ", clear to send"))

In [ ]:
# ---- Queue the batch: claim the runs, log the mail, hand the payload to the pipeline ----
# Every picked failure goes in this one body; nothing is claimed unless the mail goes out.
subject = alert_subject(due)
body    = alert_html(due, MAIL_MAX_RUNS_IN_BODY, last_mail_at)
payload = {"alert_count": 0, "subject": "", "message_html": "", "mail_batch_id": "", "recipients": ALERT_RECIPIENTS}

if due and not MAIL_ENABLED:
    # Preview: show what would go out, touch nothing.
    print(f"MAIL_ENABLED = False: preview only, nothing claimed, logged or sent\n\nSubject: {subject}\n")
    try:
        displayHTML(body)
    except NameError:
        print(body)

elif due and in_cooldown:
    # Inside the window: leave them unclaimed so they ride along in the next digest.
    print(f"{len(due)} failure(s) held: inside the {MAIL_COOLDOWN_MINUTES} min window since"
          f" {fmt_local(last_mail_at)}. Next mail from {fmt_local(cooldown_until)}, and they go in it.")

elif due:
    batch_id = str(uuid.uuid4())
    queued_status = "SentUnconfirmed" if ASSUME_SENT_ON_EXIT else "Queued"

    # Claim the runs before the payload leaves, so a second run of this notebook cannot
    # mail the same failure twice. first_queued_at_utc is insert-only.
    claims = [(r.run_key, r.source_type, r.item_name, r.effective_status, r.run_end_time_utc,
               ASSUME_SENT_ON_EXIT, now if ASSUME_SENT_ON_EXIT else None,
               queued_status, r.mail_attempt_count + 1, batch_id, ALERT_RECIPIENTS or None, None,
               now, now, now)
              for r in due]
    spark.createDataFrame(claims, spark.table(STATE_TABLE).schema) \
         .dropDuplicates(["run_key"]).createOrReplaceTempView("stg_alert_claims")

    spark.sql(f"""
        MERGE INTO {STATE_TABLE} AS t
        USING stg_alert_claims AS s
           ON t.run_key = s.run_key
        WHEN MATCHED THEN UPDATE SET
            effective_status   = s.effective_status,
            run_end_time_utc   = s.run_end_time_utc,
            sent_mail          = s.sent_mail,
            sent_mail_at_utc   = s.sent_mail_at_utc,
            mail_status        = s.mail_status,
            mail_attempt_count = s.mail_attempt_count,
            mail_batch_id      = s.mail_batch_id,
            mail_recipients    = s.mail_recipients,
            mail_error         = NULL,
            last_queued_at_utc = s.last_queued_at_utc,
            updated_at_utc     = s.updated_at_utc
        WHEN NOT MATCHED THEN INSERT *
    """)

    spark.createDataFrame(
        [(str(uuid.uuid4()), batch_id, queued_status, len(due),
          ", ".join(sorted({r.source_type for r in due})),
          ", ".join(r.run_key for r in due),
          ALERT_RECIPIENTS or None, subject, body,
          now, now if ASSUME_SENT_ON_EXIT else None, None, notebook_run_id(), now)],
        spark.table(MAIL_LOG_TABLE).schema,
    ).write.insertInto(MAIL_LOG_TABLE)

    payload = {"alert_count": len(due), "subject": subject, "message_html": body,
               "mail_batch_id": batch_id, "recipients": ALERT_RECIPIENTS}

    print(f"Batch {batch_id}: {len(due)} run(s) {queued_status} for {ALERT_RECIPIENTS or 'the recipients the pipeline sends to'}")
    print(f"Subject: {subject}")
    display(spark.sql(f"""
        SELECT run_key, source_type, item_name, effective_status,
               sent_mail, mail_status, mail_attempt_count, mail_recipients
        FROM {STATE_TABLE}
        WHERE mail_batch_id = {sq(batch_id)}
        ORDER BY source_type, item_name
    """))

else:
    print("No failures to mail")

mssparkutils.notebook.exit(json.dumps(payload))

## Checks

The cells below never run in the pipeline - `notebook.exit` above ends the run - they are for looking at the state by hand.

In [ ]:
# ---- Last 2 days: every failure and whether anyone was told ----
display(spark.sql(f"""
    SELECT source_type, item_name, run_name, start_time_sgt, effective_status,
           duration_minutes, child_failed, failed_children, error_detail,
           sent_mail, sent_mail_at_sgt, mail_status, mail_attempt_count, mail_error
    FROM {STATUS_VIEW}
    WHERE run_date_sgt >= date_sub(current_date(), 2)
    ORDER BY start_time_sgt DESC
"""))

In [ ]:
# ---- Mail log, newest first ----
display(spark.sql(f"""
    SELECT from_utc_timestamp(queued_at_utc, '{LOCAL_TZ}') AS queued_at_sgt,
           from_utc_timestamp(sent_at_utc,   '{LOCAL_TZ}') AS sent_at_sgt,
           mail_status, run_count, source_types, recipients, subject,
           error_message, mail_batch_id, notebook_run_id
    FROM {MAIL_LOG_TABLE}
    ORDER BY queued_at_utc DESC
    LIMIT 50
"""))

In [ ]:
# ---- Anything that never got out: unsent, timed out, or given up on ----
display(spark.sql(f"""
    SELECT source_type, item_name, effective_status,
           from_utc_timestamp(run_end_time_utc, '{LOCAL_TZ}') AS run_end_sgt,
           mail_status, mail_attempt_count, mail_error, mail_batch_id, run_key
    FROM {STATE_TABLE}
    WHERE NOT sent_mail
    ORDER BY run_end_time_utc DESC
"""))